# Fig 11 and Fig 12 - Qwen across Lancaster dimensions and by dominant label
Fig 11: correlations between Qwen's predictions and each of the 11 Lancaster dimensions, alongside the human Interoceptive baseline.
Fig 12: Qwen performance grouped by `Dominant.sensorimotor` label.

## Load Lancaster norms + Qwen predictions

In [ ]:
from data_loading import DATA_DIR
import json, pandas as pd
lan = pd.read_csv(DATA_DIR / "Lancaster_sensorimotor_norms_for_39707_words.csv")
lan = lan.rename(columns={"Word": "WORD"})

qwen_path = DATA_DIR / "prediction_qwen_temp0_full_data_set.jsonl"
with open(qwen_path) as f:
    q = pd.DataFrame(json.loads(f.read()))[["WORD", "expected_score"]].rename(
        columns={"expected_score": "qwen_expected"}
    )

df = lan.merge(q, on="WORD", how="inner")
print(f"Merged rows: {len(df):,}")
df.head()


## Fig 11 - Qwen vs. each Lancaster dimension

In [ ]:
from scipy.stats import pearsonr

perceptual = ["Visual.mean", "Auditory.mean", "Haptic.mean",
              "Olfactory.mean", "Gustatory.mean", "Interoceptive.mean"]
action     = ["Foot_leg.mean", "Hand_arm.mean", "Head.mean", "Mouth.mean", "Torso.mean"]
dims       = perceptual + action

rows = []
for d in dims:
    cols = list(dict.fromkeys(["qwen_expected", "Interoceptive.mean", d]))
    sub = df[cols].dropna()
    r_q,  _ = pearsonr(sub["qwen_expected"],         sub[d])
    r_h,  _ = pearsonr(sub["Interoceptive.mean"],    sub[d])
    rows.append({
        "Dimension": d.replace(".mean", "").replace("_", "/"),
        "Qwen Pearson":  r_q,
        "Human-Interoceptive Pearson":  r_h,
    })

corr = pd.DataFrame(rows).sort_values("Qwen Pearson", ascending=False).reset_index(drop=True)
corr


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dims_lbl = corr["Dimension"].tolist()
qwen = corr["Qwen Pearson"].values
hum  = corr["Human-Interoceptive Pearson"].values

fig, ax = plt.subplots(figsize=(13, 6), dpi=200)
x = np.arange(len(dims_lbl)); w = 0.4
qc = ["#9467bd" if d == "Interoceptive" else "#cdb4db" for d in dims_lbl]
hc = ["#1f77b4" if d == "Interoceptive" else "#a8c8e8" for d in dims_lbl]

b1 = ax.bar(x - w/2, qwen, w, label="Qwen vs. human dimension",
            color=qc, edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + w/2, hum,  w, label="Human Interoceptive vs. human dimension",
            color=hc, edgecolor="black", linewidth=0.5)
for bar, v in zip(b1, qwen):
    y = v + 0.02 if v >= 0 else v - 0.05
    ax.text(bar.get_x()+bar.get_width()/2, y, f"{v:.2f}", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=9)
for bar, v in zip(b2, hum):
    y = v + 0.02 if v >= 0 else v - 0.05
    ax.text(bar.get_x()+bar.get_width()/2, y, f"{v:.2f}", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=9)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(dims_lbl, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Pearson correlation", fontsize=11)
ax.set_title("Qwen's interoceptive predictions vs. each Lancaster dimension,\n"
             "alongside the human Interoceptive baseline", fontsize=12, fontweight="bold")
ax.set_ylim(-0.5, 1.15)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=True, fontsize=10, loc="upper right")
plt.tight_layout()
plt.savefig("fig11_qwen_lancaster_dimensions.png", dpi=200, bbox_inches="tight")
plt.show()


## Fig 12 - Qwen by dominant sensorimotor label

In [ ]:
df["error_signed"] = df["qwen_expected"] - df["Interoceptive.mean"]
df["error_abs"]    = df["error_signed"].abs()

rows = []
for label, sub in df.groupby("Dominant.sensorimotor"):
    if len(sub) < 2: continue
    r, _ = pearsonr(sub["Interoceptive.mean"], sub["qwen_expected"])
    rows.append({
        "Label": label.replace("_", "/"),
        "N": len(sub),
        "Mean Human (Interoceptive)": sub["Interoceptive.mean"].mean(),
        "Mean Qwen":                  sub["qwen_expected"].mean(),
        "Mean signed error":          sub["error_signed"].mean(),
        "Mean abs error":             sub["error_abs"].mean(),
        "Pearson r (Qwen, Human)":    r,
    })

by_label = pd.DataFrame(rows).sort_values("Mean Human (Interoceptive)",
                                          ascending=False).reset_index(drop=True)
by_label


In [ ]:
labels = by_label["Label"].tolist()
ns     = by_label["N"].astype(int).tolist()
human  = by_label["Mean Human (Interoceptive)"].values
qwen   = by_label["Mean Qwen"].values
sgn    = by_label["Mean signed error"].values
pr     = by_label["Pearson r (Qwen, Human)"].values

x = np.arange(len(labels)); w = 0.4
fig, axes = plt.subplots(2, 1, figsize=(14, 9.5), dpi=200, sharex=True)

ax = axes[0]
b1 = ax.bar(x - w/2, human, w, label="Human Interoceptive (mean rating)",
            color="#1f77b4", edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + w/2, qwen,  w, label="Qwen prediction (mean expected score)",
            color="#9467bd", edgecolor="black", linewidth=0.5)
for bar, v in zip(b1, human):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.04, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9)
for bar, v in zip(b2, qwen):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.04, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Mean interoceptive score (0-5)", fontsize=11)
ax.set_title("Mean human interoceptive rating and Qwen prediction, per dominant sensorimotor label",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, max(human.max(), qwen.max()) + 0.5)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=True, fontsize=10, loc="upper left")

ax = axes[1]
b3 = ax.bar(x - w/2, sgn, w, label="Mean signed error (Qwen - Human)",
            color="#d62728", edgecolor="black", linewidth=0.5)
b4 = ax.bar(x + w/2, pr,  w, label="Pearson r (Qwen vs. Human, within label)",
            color="#2ca02c", edgecolor="black", linewidth=0.5)
for bar, v in zip(b3, sgn):
    y = v + 0.025 if v >= 0 else v - 0.05
    ax.text(bar.get_x()+bar.get_width()/2, y, f"{v:+.2f}", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=9)
for bar, v in zip(b4, pr):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.025, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("Error / correlation", fontsize=11)
ax.set_title("Qwen accuracy per dominant sensorimotor label",
             fontsize=12, fontweight="bold")
ax.set_ylim(min(sgn.min(), pr.min()) - 0.25, max(sgn.max(), pr.max()) + 0.30)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=True, fontsize=10, loc="upper left")
ax.set_xticks(x)
ax.set_xticklabels([f"{l}\nn = {n:,}" for l, n in zip(labels, ns)], fontsize=9)
plt.tight_layout()
plt.savefig("fig12_qwen_by_label.png", dpi=200, bbox_inches="tight")
plt.show()
